# Inspect the OCDS JSON structure

Examines the monthly SERCOP release packages, documents the OCDS fields used by the project, and establishes the raw-data inventory.

Run this notebook from the `notebooks/` directory after completing the preceding numbered stage. Generated files are written to the documented project directories.


In [ ]:
import json
from pathlib import Path

archivo_enero = Path("../data/raw/2025/sercop_2025_january_electronic_reverse_auction.json")

with open(archivo_enero, "r", encoding="utf-8") as archivo:
    datos_enero = json.load(archivo)

type(datos_enero)


In [ ]:
len(datos_enero)


#### Identificar el tipo de cada registro

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
type(datos_enero[0])


#### Identificar los campos disponibles

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
datos_enero[0].keys()


#### Explorar la lista de releases

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
type(datos_enero[0]["releases"])


In [ ]:
len(datos_enero[0]["releases"])


#### Identificar los campos del primer release

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
datos_enero[0]["releases"][0].keys()


#### Explorar la entidad compradora

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
datos_enero[0]["releases"][0]["buyer"]


This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
datos_enero[0]["releases"][0]["tender"].keys()


#### Revisar los valores principales del procedimiento

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
tender = datos_enero[0]["releases"][0]["tender"]

{
    "id": tender.get("id"),
    "title": tender.get("title"),
    "status": tender.get("status"),
    "procurementMethod": tender.get("procurementMethod"),
    "procurementMethodDetails": tender.get("procurementMethodDetails"),
    "numberOfTenderers": tender.get("numberOfTenderers")
}


#### Explorar los oferentes del procedimiento

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
type(tender["tenderers"]), len(tender["tenderers"])


#### Identificar los campos de los oferentes

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
tender["tenderers"][0].keys()


#### Visualizar los oferentes participantes

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
tender["tenderers"]


#### Explorar las partes relacionadas

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
release = datos_enero[0]["releases"][0]

type(release["parties"]), len(release["parties"])


#### Identificar la estructura de las partes

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
release["parties"][0].keys()


#### Identificar los roles de las organizaciones

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
[
    {
        "id": parte.get("id"),
        "name": parte.get("name"),
        "roles": parte.get("roles")
    }
    for parte in release["parties"]
]


This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
claves_releases = set()

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        claves_releases.update(release_item.keys())

sorted(claves_releases)


This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
releases_con_awards = []

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        if release_item.get("awards"):
            releases_con_awards.append(release_item)

len(releases_con_awards)


#### Explorar la estructura de las adjudicaciones

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
releases_con_awards[0]["awards"]


This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
award = releases_con_awards[0]["awards"][0]

{
    "award_id": award.get("id"),
    "fecha": award.get("date"),
    "monto": award.get("value", {}).get("amount"),
    "moneda": award.get("value", {}).get("currency"),
    "proveedores": award.get("suppliers", [])
}


This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
todos_los_releases = [
    release_item
    for paquete in datos_enero
    for release_item in paquete.get("releases", [])
]

ocids = [
    release_item.get("ocid")
    for release_item in todos_los_releases
    if release_item.get("ocid")
]

{
    "total_releases": len(todos_los_releases),
    "procedimientos_unicos": len(set(ocids)),
    "releases_repetidos": len(ocids) - len(set(ocids))
}


#### Contar proveedores adjudicados por procedimiento

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
cantidad_proveedores_por_award = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        cantidad_proveedores_por_award.append(
            len(award_item.get("suppliers", []))
        )

{
    "total_awards": len(cantidad_proveedores_por_award),
    "minimo_proveedores": min(cantidad_proveedores_por_award),
    "maximo_proveedores": max(cantidad_proveedores_por_award),
    "distribucion": {
        cantidad: cantidad_proveedores_por_award.count(cantidad)
        for cantidad in sorted(set(cantidad_proveedores_por_award))
    }
}


#### Analizar los montos adjudicados

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


In [ ]:
montos_adjudicados = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        monto = award_item.get("value", {}).get("amount")

        if monto is not None:
            montos_adjudicados.append(monto)

{
    "cantidad_montos": len(montos_adjudicados),
    "monto_minimo": min(montos_adjudicados),
    "monto_maximo": max(montos_adjudicados),
    "monto_promedio": sum(montos_adjudicados) / len(montos_adjudicados),
    "monto_total": sum(montos_adjudicados)
}


In [ ]:
from pathlib import Path
import json
import pandas as pd

rutas_posibles = [
    Path.cwd() / "data" / "raw" / "2025",
    Path.cwd().parent / "data" / "raw" / "2025"
]

carpeta_datos = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if carpeta_datos is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/raw/2025."
    )

archivos = sorted(carpeta_datos.glob("*.json"))

print("Carpeta encontrada:", carpeta_datos)
print("Número de archivos JSON:", len(archivos))

if len(archivos) != 12:
    print("ADVERTENCIA: Se esperaban 12 archivos mensuales.")


ocids = set()
entidades = set()
oferentes = set()
proveedores_adjudicados = set()
procedimientos_con_adjudicacion = set()
adjudicaciones = set()

total_releases = 0
resumen_mensual = []


def identificador_actor(actor):
    """Obtiene el identificador; usa el nombre solo si no existe ID."""
    if not isinstance(actor, dict):
        return None

    return actor.get("id") or actor.get("name")


for archivo in archivos:
    with open(archivo, "r", encoding="utf-8") as entrada:
        contenido = json.load(entrada)

    paquetes = contenido if isinstance(contenido, list) else [contenido]

    releases_archivo = 0
    ocids_archivo = set()

    for paquete in paquetes:
        if not isinstance(paquete, dict):
            continue

        releases = paquete.get("releases", [])

        if not releases and paquete.get("ocid"):
            releases = [paquete]

        for release in releases:
            total_releases += 1
            releases_archivo += 1

            ocid = release.get("ocid")
            if ocid:
                ocids.add(ocid)
                ocids_archivo.add(ocid)

            tender = release.get("tender") or {}

            # Entidad compradora o contratante
            for actor in [
                release.get("buyer"),
                tender.get("procuringEntity")
            ]:
                actor_id = identificador_actor(actor)
                if actor_id:
                    entidades.add(actor_id)

            # Oferentes registrados en tender
            for oferente in tender.get("tenderers", []) or []:
                oferente_id = identificador_actor(oferente)
                if oferente_id:
                    oferentes.add(oferente_id)

            # Roles registrados en parties
            for participante in release.get("parties", []) or []:
                participante_id = identificador_actor(participante)
                roles = participante.get("roles", []) or []

                if participante_id and (
                    "buyer" in roles or "procuringEntity" in roles
                ):
                    entidades.add(participante_id)

                if participante_id and "tenderer" in roles:
                    oferentes.add(participante_id)

            for adjudicacion in release.get("awards", []) or []:
                if ocid:
                    procedimientos_con_adjudicacion.add(ocid)

                award_id = adjudicacion.get("id")

                if award_id:
                    adjudicaciones.add((ocid, award_id))

                for proveedor in adjudicacion.get("suppliers", []) or []:
                    proveedor_id = identificador_actor(proveedor)

                    if proveedor_id:
                        proveedores_adjudicados.add(proveedor_id)

    resumen_mensual.append({
        "archivo": archivo.name,
        "releases": releases_archivo,
        "procedimientos_unicos_ocid": len(ocids_archivo)
    })


tabla_mensual = pd.DataFrame(resumen_mensual)
display(tabla_mensual)


resumen_general = {
    "archivos_json": len(archivos),
    "releases_totales": total_releases,
    "procedimientos_unicos_ocid": len(ocids),
    "entidades_identificadas": len(entidades),
    "oferentes_unicos": len(oferentes),
    "procedimientos_con_adjudicacion": len(
        procedimientos_con_adjudicacion
    ),
    "adjudicaciones_unicas": len(adjudicaciones),
    "proveedores_adjudicados_unicos": len(
        proveedores_adjudicados
    )
}

display(
    pd.DataFrame.from_dict(
        resumen_general,
        orient="index",
        columns=["cantidad"]
    )
)


# 6. Generar dos párrafos preliminares
print(
    f"\nEl conjunto inicial de datos estuvo integrado por "
    f"{len(archivos)} archivos mensuales en formato JSON. "
    f"En total se identificaron {total_releases:,} releases, "
    f"correspondientes a {len(ocids):,} procedimientos únicos "
    f"según el identificador ocid."
)

print(
    f"\nEn la revisión preliminar se reconocieron "
    f"{len(entidades):,} entidades contratantes y "
    f"{len(oferentes):,} oferentes. Asimismo, "
    f"{len(procedimientos_con_adjudicacion):,} procedimientos "
    f"presentaron información de adjudicación, con "
    f"{len(proveedores_adjudicados):,} proveedores adjudicados "
    f"identificados."
)


In [ ]:
%pip install pandas


In [ ]:
import json

ruta = "../data/raw/2025/sercop_2025_january_electronic_reverse_auction.json"

with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

type(datos)


In [ ]:
len(datos)


In [ ]:
datos[0].keys()


In [ ]:
len(datos[0]["releases"])


In [ ]:
datos[0]["releases"][0].keys()


In [ ]:
datos[0]["releases"][0]["ocid"]


In [ ]:
datos[0]["releases"][0]["buyer"]


In [ ]:
datos[0]["releases"][0]["tender"].keys()


In [ ]:
datos[0]["releases"][0]["tender"]["numberOfTenderers"]


In [ ]:
len(datos[0]["releases"][0]["tender"]["tenderers"])


In [ ]:
datos[0]["releases"][0]["tender"]["tenderers"]


In [ ]:
datos[0]["releases"][0].keys()


In [ ]:
"awards" in datos[0]["releases"][0]


In [ ]:
for i, paquete in enumerate(datos):
    release = paquete["releases"][0]
    if "awards" in release:
        print("Índice:", i)
        print("OCID:", release["ocid"])
        break


In [ ]:
datos[2]["releases"][0]["awards"]


In [ ]:
datos[2]["releases"][0]["awards"][0]["items"][0]["classification"]


In [ ]:
datos[2]["releases"][0]["tender"]["lots"]


In [ ]:
datos[2]["releases"][0]["date"]


In [ ]:
datos[2]["releases"][0]["parties"][:2]
